# TMS Aorta: verified load and compact EDA

This notebook downloads the pinned Tabula Muris Senis Aorta H5AD into a caller-selected cache, or reuses a manifest supplied through `BIOML_ARTIFACT_MANIFEST`. It then performs a sparse-safe Scanpy QC pass and summarizes donor and cell-type coverage.

Source: Tabula Muris Senis Data Objects, Figshare article 12654728, file 23872460, DOI [10.6084/m9.figshare.12654728.v1](https://doi.org/10.6084/m9.figshare.12654728.v1).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from IPython.display import display as show
from scipy import sparse

import bioml_data as bio

## Resolve and verify the artifact

For normal interactive use, set `BIOML_DATA_DIR` to the desired cache directory. Automated or offline runs can set `BIOML_ARTIFACT_MANIFEST` to an existing verified manifest.

In [ ]:
data_dir = Path(os.environ.get("BIOML_DATA_DIR", ".cache/bioml-data"))
manifest_value = os.environ.get("BIOML_ARTIFACT_MANIFEST")
if manifest_value:
    artifact = bio.load_artifact_receipt(Path(manifest_value))
else:
    artifact = bio.download_dataset("tms-aorta", data_dir=data_dir).artifact

adata = bio.load_anndata(artifact)
canonical_manifest = os.environ.get("BIOML_CANONICAL_ARTIFACT_MANIFEST")
if canonical_manifest:
    canonical_artifact = bio.load_artifact_receipt(Path(canonical_manifest))
else:
    canonical_artifact = bio.prepare_dataset(
        "tms-aorta", artifact=artifact, data_dir=data_dir
    ).artifact
canonical = bio.load_dataset("tms-aorta", artifact=canonical_artifact)
assert canonical.counts.shape.observations == adata.n_obs
assert canonical.counts.shape.features == adata.n_vars
assert sparse.issparse(adata.X)
assert adata.obs_names.is_unique
assert adata.var_names.is_unique
assert {"mouse.id", "cell_ontology_class", "donor_id", "cell_type"}.issubset(
    adata.obs.columns
)
assert (
    adata.obs["mouse.id"].astype(str).to_numpy()
    == adata.obs["donor_id"].astype(str).to_numpy()
).all()
assert (
    adata.obs["cell_ontology_class"].astype(str).to_numpy()
    == adata.obs["cell_type"].astype(str).to_numpy()
).all()
adata

## QC and cohort coverage

The QC pass computes per-cell total counts and detected genes without converting the expression matrix to a dense array.

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
donor_counts = adata.obs["donor_id"].value_counts().sort_values(ascending=False)
cell_type_counts = adata.obs["cell_type"].value_counts().sort_values(ascending=False)

print(f"shape: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"donors: {donor_counts.size:,}")
print(f"cell types: {cell_type_counts.size:,}")
show(adata.obs[["total_counts", "n_genes_by_counts"]].describe())
show(donor_counts.to_frame("cells"))
show(cell_type_counts.to_frame("cells"))

## Save headless EDA artifacts

The same cell works interactively and in CI. `BIOML_EDA_OUTPUT_DIR` controls where the summary and figure are written.

In [ ]:
output_dir = Path(os.environ.get("BIOML_EDA_OUTPUT_DIR", "artifacts/tms-aorta-eda"))
output_dir.mkdir(parents=True, exist_ok=True)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].hist(np.log1p(adata.obs["total_counts"].to_numpy()), bins=30, color="#4C78A8")
axes[0].set(
    title="Library size distribution", xlabel="log1p total counts", ylabel="Cells"
)

top_cell_types = cell_type_counts.head(12).sort_values()
axes[1].barh(
    top_cell_types.index.astype(str), top_cell_types.to_numpy(), color="#59A14F"
)
axes[1].set(title="Most abundant cell types", xlabel="Cells")

figure_path = output_dir / "tms_aorta_eda.png"
figure.savefig(figure_path, dpi=160)
plt.close(figure)

summary = {
    "cells": int(adata.n_obs),
    "genes": int(adata.n_vars),
    "donors": int(donor_counts.size),
    "cell_types": int(cell_type_counts.size),
    "sparse": bool(sparse.issparse(adata.X)),
}
summary_path = output_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary